In [3]:
# from .autonotebook import tqdm as notebook_tqdm
from huggingface_hub import snapshot_download
from pathlib import Path

mistral_models_path = Path.home().joinpath('mistral_models', '7B-Instruct-v0.3')
mistral_models_path.mkdir(parents=True, exist_ok=True)

snapshot_download(repo_id="mistralai/Mistral-7B-Instruct-v0.3", allow_patterns=["params.json", "consolidated.safetensors", "tokenizer.model.v3"], local_dir=mistral_models_path)


Fetching 3 files: 100%|██████████| 3/3 [40:57<00:00, 819.00s/it]   


'/root/mistral_models/7B-Instruct-v0.3'

In [1]:
from mistral_inference.transformer import Transformer
from mistral_inference.generate import generate

from mistral_common.tokens.tokenizers.mistral import MistralTokenizer
from mistral_common.protocol.instruct.messages import UserMessage
from mistral_common.protocol.instruct.request import ChatCompletionRequest


tokenizer = MistralTokenizer.from_file(f"{mistral_models_path}/tokenizer.model.v3")
model = Transformer.from_folder(mistral_models_path)

completion_request = ChatCompletionRequest(messages=[UserMessage(content="Explain Machine Learning to me in a nutshell.")])

tokens = tokenizer.encode_chat_completion(completion_request).tokens

out_tokens, _ = generate([tokens], model, max_tokens=64, temperature=0.0, eos_id=tokenizer.instruct_tokenizer.tokenizer.eos_id)
result = tokenizer.instruct_tokenizer.tokenizer.decode(out_tokens[0])

print(result)


NameError: name 'mistral_models_path' is not defined

In [9]:
from gliner import GLiNER
from peft import LoraConfig, get_peft_model, TaskType
from gliner.data_processing.collator import DataCollator
from gliner.training import Trainer, TrainingArguments

model = GLiNER.from_pretrained("knowledgator/modern-gliner-bi-large-v1.0")


display(model)


Fetching 9 files: 100%|██████████| 9/9 [00:00<00:00, 198677.56it/s]


GLiNER(
  (model): SpanModel(
    (token_rep_layer): BiEncoder(
      (bert_layer): Transformer(
        (model): ModernBertModel(
          (embeddings): ModernBertEmbeddings(
            (tok_embeddings): Embedding(50368, 1024, padding_idx=50283)
            (norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
            (drop): Dropout(p=0.0, inplace=False)
          )
          (layers): ModuleList(
            (0): ModernBertEncoderLayer(
              (attn_norm): Identity()
              (attn): ModernBertAttention(
                (Wqkv): Linear(in_features=1024, out_features=3072, bias=False)
                (rotary_emb): ModernBertRotaryEmbedding()
                (Wo): Linear(in_features=1024, out_features=1024, bias=False)
                (out_drop): Identity()
              )
              (mlp_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
              (mlp): ModernBertMLP(
                (Wi): Linear(in_features=1024, out_features=5248, bias=F

In [12]:
model.config

GLiNERConfig {
  "class_token_index": -1,
  "dropout": 0.35,
  "embed_ent_token": true,
  "encoder_config": {
    "_attn_implementation_autoset": false,
    "_name_or_path": "answerdotai/ModernBERT-large",
    "add_cross_attention": false,
    "architectures": [
      "ModernBertForMaskedLM"
    ],
    "attention_bias": false,
    "attention_dropout": 0.0,
    "bad_words_ids": null,
    "begin_suppress_tokens": null,
    "bos_token_id": 50281,
    "chunk_size_feed_forward": 0,
    "classifier_activation": "gelu",
    "classifier_bias": false,
    "classifier_dropout": 0.0,
    "classifier_pooling": "mean",
    "cls_token_id": 50281,
    "cross_attention_hidden_size": null,
    "decoder_bias": true,
    "decoder_start_token_id": null,
    "deterministic_flash_attn": false,
    "diversity_penalty": 0.0,
    "do_sample": false,
    "dtype": "float32",
    "early_stopping": false,
    "embedding_dropout": 0.0,
    "encoder_no_repeat_ngram_size": 0,
    "eos_token_id": 50282,
    "exponenti

In [10]:
model.config.max_len



2048

In [11]:
model.data_processor.transformer_tokenizer.model_max_length

1000000000000000019884624838656

In [7]:
model.data_processor.transformer_tokenizer.model_max_length

8192